# BikeNodePlanner–dk_bicycle_network LTS combination
The main workflow keeps the original sample-point matching loop. `SAMPLING_LENGTH_M` controls the preview, segment split, and segment map; `MAX_LTS_PLOT_SETTING` independently controls the maximum-per-link map.

The full `SAMPLING_LENGTH_M_LIST` is used only in the comparison-table cell near the end. The regional-table update is in its own cell. Only the maximum map and selected segment map are saved. The original `combine_denmark.ipynb` is not changed.

Environment: the unchanged spatial-index loop needs a GeoPandas version supporting `query(..., predicate="dwithin", distance=...)`. This copy was verified with GeoPandas 1.1.3; GeoPandas 0.14.4 in the older `dk_bike_network` environment does not support the `distance` argument. Select a compatible notebook kernel.


## Setup

In [ ]:
from pathlib import Path
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import shapely
from IPython.display import display
from shapely.geometry import box

working_dir = Path.cwd().resolve()

PROJECT_ROOT = working_dir.parent if working_dir.name == "combine" else working_dir  # this notebook lives in bike_safety/combine/

# Output folders for this notebook
COMBINE_DIR = PROJECT_ROOT / "combine"
DATA_OUTPUT_DIR = COMBINE_DIR / "data"
PLOTS_OUTPUT_DIR = COMBINE_DIR / "plots"

# DK_bicycle_network with LTS, produced by dk_bicycle_network (script 11)
DK_NETWORK_EDGES_PATH = PROJECT_ROOT / "dk_bicycle_network/data/processed/osm_road_edges.parquet"

EXTRACTION_MARGIN_M = 500  # only needs to exceed TOLERANCE_M
TOLERANCE_M = 10           # max distance between a bike-loop link and an OSM edge

SAMPLING_LENGTH_M_LIST = ["include all segments", 0.1, 0.5, 1, 2, 5, 10, 15, 20, 50, 100, 200]
SAMPLING_LENGTH_M = 2      # preview and segment-map spacing, in metres

LENGTH_THRESHOLD_LIST = [0, 1, 2, 5, 10, 15, 20, 50]  # minimum length of a LTS edge to be included in the analysis (in meters)
LENGTH_THRESHOLD = 2       # minimum length of a LTS edge to be included in the analysis (in meters)

SHOW_INTERACTIVE_MAP = True  # display the selected result interactively

# Columns needed from the dk_bicycle_network edges
OSM_COLUMNS = [ "lts", "cycling_allowed", "geometry"]

# LTS 999 = no cycling (pedestrian paths). LTS 0 = car-free paths where cycling is allowed.
EXCLUDED_LTS = [999]
LTS_0_AS = 1  # research choice: treat car-free paths as low stress

### Plot helpers
Shared by the three LTS maps (worst LTS per link, length-weighted LTS, dk_bicycle_network ground truth). Each map gets a bar chart as its legend: km per LTS class plus "not covered", with the share of the total km inside each bar. Map and bars use the same colours.

In [ ]:
LTS_CLASSES = [1, 2, 3, 4]
LTS_CMAP = plt.get_cmap("RdYlGn_r")  # 1 = green, 4 = red
LTS_COLORS = {lts: LTS_CMAP((lts - 1) / 3) for lts in LTS_CLASSES}
NOT_COVERED_COLOR = "black"


def km_per_lts_class(lts_values, lengths_m):
    """Total km per LTS class 1–4, plus "not covered" for missing LTS (NaN)."""
    lts_values = pd.Series(np.asarray(lts_values, dtype=float))
    lengths_km = pd.Series(np.asarray(lengths_m, dtype=float)) / 1000
    km = {str(lts): lengths_km[lts_values == lts].sum() for lts in LTS_CLASSES}
    km["not covered"] = lengths_km[lts_values.isna()].sum()
    return pd.Series(km)


def percentages_summing_to_100(values, decimals=0):
    """Percentages of `values` that add up to exactly 100 (largest remainder method).

    Every percentage is rounded down (to `decimals` places), and the missing steps go to the values
    with the largest remainders, so no bar is off by a full step and the bars always sum to 100%.
    """
    values = np.asarray(values, dtype=float)
    total = values.sum()
    if total <= 0:
        return np.zeros(len(values), dtype=int)
    steps = 10 ** decimals
    exact = 100 * steps * values / total
    rounded = np.floor(exact).astype(int)
    missing = 100 * steps - rounded.sum()
    rounded[np.argsort(-(exact - rounded), kind="stable")[:missing]] += 1
    return rounded / steps if decimals else rounded


def add_lts_bar_legend(ax, km, title, xlabel="LTS", bounds=(0.66, 0.58, 0.32, 0.36),
                       colors=None, tick_labels=None):
    """Bar chart inside the map: km per class, with the percentage of the total in each bar.

    Defaults to LTS 1–4 + "not covered"; pass `colors` and `tick_labels` for other bars (e.g. a histogram).
    """
    inset = ax.inset_axes(bounds)
    if colors is None:
        colors = [LTS_COLORS[lts] for lts in LTS_CLASSES] + [NOT_COVERED_COLOR]
    if tick_labels is None:
        tick_labels = ["1", "2", "3", "4", "not\ncovered"]
    x = np.arange(len(km))
    bars = inset.bar(x, km.to_numpy(), width=1.0, color=colors)

    y_max = km.max() * 1.08 if km.max() > 0 else 1
    inset.set_ylim(0, y_max)
    for bar, value, color, pct in zip(bars, km, colors, percentages_summing_to_100(km)):
        # White text on dark bars; black text on light bars and on bars too short to hold the text
        r, g, b = plt.matplotlib.colors.to_rgb(color)
        dark_bar = (0.299 * r + 0.587 * g + 0.114 * b) < 0.5 and value > 0.12 * y_max
        inset.text(
            bar.get_x() + bar.get_width() / 2, 0.03 * y_max, f"{pct}%",
            ha="center", va="bottom", fontsize=9, color="white" if dark_bar else "black",
        )

    inset.set_xticks(x, tick_labels, fontsize=8 if len(x) > 5 else None)
    inset.yaxis.set_major_formatter(plt.matplotlib.ticker.StrMethodFormatter("{x:,.0f}"))  # 17,500 not 17500
    inset.set_xlabel(xlabel)
    inset.set_ylabel("km")
    inset.set_title(title, fontsize=11)
    inset.set_facecolor("white")
    return inset


def add_lts_histogram_legend(ax, bin_edges, km_per_bin, not_covered_km, title, xlabel,
                             bounds=(0.60, 0.55, 0.38, 0.40)):
    """Histogram inside the map for a continuous LTS score (1–4): km per bin, coloured like the map
    at each bin's midpoint, plus a separate "not covered" bar. Percentages (of all km, summing to 100)
    are written inside the bars (vertically when there are many bins); empty bins get no label."""
    inset = ax.inset_axes(bounds)
    bin_edges = np.asarray(bin_edges, dtype=float)
    widths = np.diff(bin_edges)
    midpoints = bin_edges[:-1] + widths / 2
    not_covered_x = bin_edges[-1] + 0.35
    not_covered_width = 0.2

    x = np.r_[midpoints, not_covered_x]
    heights = np.r_[np.asarray(km_per_bin, dtype=float), not_covered_km]
    bar_widths = np.r_[widths, not_covered_width]
    colors = [LTS_CMAP((mid - 1) / 3) for mid in midpoints] + [NOT_COVERED_COLOR]
    bars = inset.bar(x, heights, width=bar_widths, color=colors)

    y_max = heights.max() * 1.08 if heights.max() > 0 else 1
    inset.set_ylim(0, y_max)
    many_bins = len(midpoints) > 15  # narrow bars: small vertical labels
    for bar, value, color, pct in zip(bars, heights, colors, percentages_summing_to_100(heights)):
        if value <= 0:
            continue
        r, g, b = plt.matplotlib.colors.to_rgb(color)
        dark_bar = (0.299 * r + 0.587 * g + 0.114 * b) < 0.5 and value > 0.25 * y_max
        inset.text(
            bar.get_x() + bar.get_width() / 2, 0.02 * y_max, f"{pct}%",
            ha="center", va="bottom", rotation=90 if many_bins else 0, fontsize=6.5 if many_bins else 7,
            color="white" if dark_bar else "black",
        )

    tick_values = np.arange(1, 4.01, 0.5)
    inset.set_xticks(
        np.r_[tick_values, not_covered_x],
        [f"{value:g}" for value in tick_values] + ["not\ncovered"],
    )
    inset.set_xlim(bin_edges[0] - 0.05, not_covered_x + not_covered_width)
    inset.yaxis.set_major_formatter(plt.matplotlib.ticker.StrMethodFormatter("{x:,.0f}"))
    inset.set_xlabel(xlabel)
    inset.set_ylabel("km")
    inset.set_title(title, fontsize=11)
    inset.set_facecolor("white")
    return inset


# bikenwloops draws Bornholm closer to the rest of Denmark (bikenwloops/parameters/config.yml: bornholm_delta).
# Used by the Denmark maps, for drawing only: km and LTS are always computed on the real geometry.
BORNHOLM_SHIFT = dict(east_of=800_000, dx=-220_000, dy=120_000)


def shift_far_east(gdf, east_of, dx, dy):
    """Copy of `gdf` where features east of x = `east_of` (Bornholm) are moved by (dx, dy). For drawing only."""
    gdf = gdf.copy()
    far_east = (gdf.geometry.representative_point().x > east_of).to_numpy()
    if far_east.any():
        gdf.loc[far_east, "geometry"] = gdf.geometry[far_east].translate(xoff=dx, yoff=dy)
    return gdf, far_east


def plot_lts_map(gdf, column, title, km, legend_title, output_path,
                 continuous=False, background=None, linewidth=1.5, legend_xlabel="LTS",
                 legend_colors=None, legend_tick_labels=None, legend=None,
                 figsize=(12, 8), widen_right=0.75, legend_bounds=None, shift=None):
    """Map of `gdf` coloured by LTS, with the km bar chart as legend. Saves to `output_path`.

    `legend`: optional function `legend(ax)` that draws a different legend (e.g. a histogram) instead.
    `widen_right`: extra empty space to the right of the network (fraction of its width) for the legend;
        0 puts the legend on top of the map, at `legend_bounds` (axes fractions: x, y, width, height).
    `shift`: e.g. BORNHOLM_SHIFT, to draw Bornholm closer to the rest of Denmark inside a circle.
    """
    fig, ax = plt.subplots(figsize=figsize, facecolor="white")

    shifted_bounds = None
    if shift is not None:
        gdf, far_east = shift_far_east(gdf, **shift)
        if far_east.any():
            shifted_bounds = gdf.geometry[far_east].total_bounds
        if background is not None:
            background, _ = shift_far_east(background, **shift)

    if background is not None:
        background.plot(ax=ax, color="#c8cdd2", linewidth=0.3, alpha=0.6, zorder=1)

    if continuous:
        gdf.loc[gdf[column].notna()].plot(ax=ax, column=column, cmap=LTS_CMAP, vmin=1, vmax=4,
                                          linewidth=linewidth, zorder=2)
    else:
        for lts in LTS_CLASSES:
            part = gdf.loc[gdf[column] == lts]
            if len(part):
                part.plot(ax=ax, color=LTS_COLORS[lts], linewidth=linewidth, zorder=2)
    not_covered = gdf.loc[gdf[column].isna()]
    if len(not_covered):
        not_covered.plot(ax=ax, color=NOT_COVERED_COLOR, linewidth=linewidth, zorder=3)

    if shifted_bounds is not None:
        x0, y0, x1, y1 = shifted_bounds
        ax.add_patch(plt.Circle(
            ((x0 + x1) / 2, (y0 + y1) / 2), 0.62 * max(x1 - x0, y1 - y0),
            fill=False, color="#444444", linewidth=0.8, zorder=4,
        ))

    if widen_right:
        # Widen the map to the right, so the bar chart sits next to the network instead of on top of it
        x_min, x_max = ax.get_xlim()
        ax.set_xlim(x_min, x_max + widen_right * (x_max - x_min))
    ax.set_title(title, loc="left", fontsize=13, weight="bold")
    ax.set_axis_off()

    if legend is not None:
        legend(ax)
    else:
        bar_kwargs = {} if legend_bounds is None else {"bounds": legend_bounds}
        add_lts_bar_legend(ax, km, legend_title, xlabel=legend_xlabel,
                           colors=legend_colors, tick_labels=legend_tick_labels, **bar_kwargs)

    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    return fig, ax

In [ ]:
# view DK_NETWORK_EDGES_PATH parquet file:
df = pd.read_parquet(DK_NETWORK_EDGES_PATH)
df.head()

In [ ]:
# find row corresponding to the shortest segment 
min_row = df.loc[df['km'].idxmin()]
# what is the length of the shortest segment in meters?
min_length_m = min_row['km'] * 1000
print(f"The shortest segment is {min_length_m:.5f} meters long.")
min_row

#### Inspect a segment

In [ ]:
"""with pd.option_context("display.max_columns", None):
    display(
        df[
            (df["lts"] == 4)
            & (df["municipality"] == "denmark")
            & (df["name"] == "Helligdomsvej")
        ]
    )"""

In [ ]:
df.describe()

## Load dk_bicycle_network edges (once for all of Denmark)

In [ ]:
### Time: ≈ 1 min
dk_edges_dk = gpd.read_parquet(DK_NETWORK_EDGES_PATH, columns=OSM_COLUMNS)

dk_edges_dk = dk_edges_dk.loc[
    dk_edges_dk["cycling_allowed"] & ~dk_edges_dk["lts"].isin(EXCLUDED_LTS)
].copy()
dk_edges_dk["lts"] = dk_edges_dk["lts"].replace(0, LTS_0_AS)

print(f"dk_bicycle_network edges with cycling allowed: {len(dk_edges_dk):,}")
print("CRS:", dk_edges_dk.crs.to_string())
display(dk_edges_dk["lts"].value_counts().sort_index().to_frame("edges"))

# denmark

In [ ]:
SAMPLE = "denmark"

BIKE_LOOPS_DIR = PROJECT_ROOT / f"bikenwloops/data/input/{SAMPLE}/network/processed"
DATA_DIR = DATA_OUTPUT_DIR / SAMPLE
PLOTS_DIR = PLOTS_OUTPUT_DIR / SAMPLE
OUTPUT_PATH = DATA_DIR / "edges_slope_lts.gpkg"
OUTPUT_SEGMENTS_PATH = DATA_DIR / "edges_lts_segments.gpkg"

DATA_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print("Bike loop network directory:", BIKE_LOOPS_DIR)
print("Data output folder:", DATA_DIR)
print("Plots output folder:", PLOTS_DIR)

In [ ]:
# Load bike loop network data
bike_loops_edges = gpd.read_file(BIKE_LOOPS_DIR / "edges.gpkg")

bike_loops_edges = bike_loops_edges.loc[
    bike_loops_edges.geometry.notna()
    & ~bike_loops_edges.geometry.is_empty
].copy()

if bike_loops_edges.crs != dk_edges_dk.crs:
    raise ValueError("Bike-loop edges and dk_bicycle_network edges have different CRS.")

print(f"Bike loop edges: {len(bike_loops_edges):,}")

### Extract area of interest from dk_bicycle_network

In [ ]:
minx, miny, maxx, maxy = bike_loops_edges.total_bounds

dk_edges = dk_edges_dk.cx[
    minx - EXTRACTION_MARGIN_M : maxx + EXTRACTION_MARGIN_M,
    miny - EXTRACTION_MARGIN_M : maxy + EXTRACTION_MARGIN_M,
].copy()

print(f"dk_bicycle_network edges in {SAMPLE}: {len(dk_edges):,}")

### Match LTS onto the bike-loop links

`match_lts_to_links` first drops OSM edges shorter than `length_threshold` (default `LENGTH_THRESHOLD`; `0` keeps everything) and then matches the remaining edges to the sample points along each link. The threshold is therefore part of every matching run.

In [ ]:
def filter_short_edges(osm_edges, length_threshold, verbose=True):
    """Keep only OSM edges that are at least `length_threshold` metres long."""
    osm_kept = osm_edges.loc[osm_edges.geometry.length >= length_threshold]

    if verbose:
        n_dropped = len(osm_edges) - len(osm_kept)
        print(
            f"Dropped {n_dropped:,} of {len(osm_edges):,} OSM edges "
            f"({100 * n_dropped / len(osm_edges):.2f}%) shorter than {length_threshold} m"
        )
    return osm_kept


def match_lts_to_links(    
    links,
    dk_edges,
    tolerance_m=20,
    sampling_length_m=20,
    length_threshold=None,   # None -> LENGTH_THRESHOLD; 0 -> keep all edges (so comparison can be made with the original dk_bicycle_network further down)
    report_every=500,
    verbose=True,
):
    
    if length_threshold is None:
        length_threshold = LENGTH_THRESHOLD

    # Drop short OSM edges before building the spatial index
    dk_edges = filter_short_edges(dk_edges, length_threshold, verbose=verbose)
    
    dk_spatial_index = dk_edges.sindex
    dk_geometries = dk_edges.geometry.to_numpy()
    dk_lts = dk_edges["lts"].to_numpy()

    max_lts_values, coverage_pcts, lts_shares = [], [], []
    point_lts_per_link = []  # LTS at every sample point, used to split links into parts
    number_of_links = len(links)

    for position, geometry in enumerate(links.geometry, start=1):
        # Sample points along the link, every sampling_length_m
        distances = np.arange(0, geometry.length, sampling_length_m)
        points = shapely.line_interpolate_point(geometry, distances)

        # Find only dk_bicycle_network lines within the matching tolerance
        candidate_positions = dk_spatial_index.query(
            geometry,
            predicate="dwithin",
            distance=tolerance_m,
        )
        candidate_positions = np.unique(candidate_positions)

        # Lowest LTS within tolerance_m of each point (inf = no match)
        point_lts = np.full(len(points), np.inf)
        if len(candidate_positions) > 0:
            candidate_geometries = dk_geometries[candidate_positions]
            is_near = shapely.dwithin(candidate_geometries[:, None], points[None, :], tolerance_m)
            candidate_lts = dk_lts[candidate_positions].astype(float)
            point_lts = np.where(is_near, candidate_lts[:, None], np.inf).min(axis=0)

        point_lts_per_link.append(point_lts)
        matched_lts = point_lts[np.isfinite(point_lts)]

        max_lts_values.append(matched_lts.max() if len(matched_lts) else np.nan)
        coverage_pcts.append(100 * len(matched_lts) / len(points))
        lts_shares.append([100 * np.mean(point_lts == lts) for lts in (1, 2, 3, 4)])

        if report_every and position % report_every == 0:
            print(f"Processed {position:,} of {number_of_links:,} bike-loop links")

    print("Matching completed.")
    return max_lts_values, coverage_pcts, lts_shares, point_lts_per_link

In [ ]:
max_lts_values, coverage_pcts, lts_shares, point_lts_per_link = match_lts_to_links(
    bike_loops_edges,
    dk_edges,
    tolerance_m=TOLERANCE_M,
    sampling_length_m=SAMPLING_LENGTH_M,
    length_threshold=LENGTH_THRESHOLD,
)

# The OSM edges that were actually used for matching (for maps and the segment comparison)
dk_edges_kept = filter_short_edges(dk_edges, LENGTH_THRESHOLD, verbose=False)

bike_loops_lts = bike_loops_edges.copy()
bike_loops_lts["max_lts"] = max_lts_values
bike_loops_lts["coverage_pct"] = coverage_pcts
bike_loops_lts[["lts1_pct", "lts2_pct", "lts3_pct", "lts4_pct"]] = lts_shares

display(bike_loops_lts[["edge_id", "mm_len", "max_lts", "coverage_pct", "lts1_pct", "lts2_pct", "lts3_pct", "lts4_pct"]].describe().round(1))
display(bike_loops_lts["max_lts"].value_counts(dropna=False).sort_index().to_frame("links"))

In [ ]:
# Layout for the Denmark maps: legend top right, on top of the map; Bornholm drawn closer, inside a circle
DENMARK_FIGSIZE = (10, 11.5)
DENMARK_LEGEND_BOUNDS = (0.61, 0.71, 0.37, 0.24)  # x, y, width, height as fractions of the map
DENMARK_WIDEN_RIGHT = 0.15  # a little empty space right of Zealand, so the legend clears the tip of Jutland

# Plot 1: bikenwloops links, sampled as above, each link coloured by its worst LTS (max_lts)
plot_lts_map(
    bike_loops_lts,
    column="max_lts",
    title=f"{SAMPLE.title()}: worst LTS per bike-loop link",
    km=km_per_lts_class(bike_loops_lts["max_lts"], bike_loops_lts.geometry.length),
    legend_title="Worst LTS per link",
    output_path=PLOTS_DIR / f"{SAMPLE}_1_max_lts.png",
    background=None,          # 2 million dk_bicycle_network edges would turn the whole country grey
    linewidth=0.6,
    figsize=DENMARK_FIGSIZE,
    widen_right=DENMARK_WIDEN_RIGHT,
    shift=BORNHOLM_SHIFT,
    legend_bounds=DENMARK_LEGEND_BOUNDS,
)

## Split links into LTS segments

The links are kept as they are, but every time the LTS changes along a link, a new segment starts. Each segment records where on the link it is (`from_m`, `to_m`, measured along the link's geometry from its start) and its LTS.

- A sample point covers `SAMPLING_LENGTH_M / 2` on either side, so a change between two points is placed halfway between them.
- The first segment starts at 0 and the last segment ends at the link's full length.
- Stretches with no dk_bicycle_network edge within `TOLERANCE_M` become segments with an empty `lts`.
- `link_id` is the link's row number in the input file. It connects each segment to its link (in `combine_denmark`, `edge_id` is not unique, so `edge_id` alone is not enough).

In [ ]:
from shapely.ops import substring

bike_loops_lts["link_id"] = bike_loops_lts.index

segments = []

for (link_id, link), point_lts in zip(bike_loops_lts.iterrows(), point_lts_per_link):
    geometry = link.geometry
    distances = np.arange(0, geometry.length, SAMPLING_LENGTH_M)

    # Positions where the LTS differs from the previous point
    change_positions = np.flatnonzero(point_lts[1:] != point_lts[:-1]) + 1
    run_starts = np.r_[0, change_positions]
    run_ends = np.r_[change_positions, len(point_lts)]  # first point after the run

    for segment_number, (start, end) in enumerate(zip(run_starts, run_ends), start=1):
        from_m = 0.0 if start == 0 else distances[start] - SAMPLING_LENGTH_M / 2
        to_m = geometry.length if end == len(point_lts) else distances[end] - SAMPLING_LENGTH_M / 2
        lts = point_lts[start]

        segments.append({
            "link_id": link_id,
            "edge_id": link["edge_id"],
            "node_start": link["node_start"],
            "node_end": link["node_end"],
            "segment": segment_number,
            "from_m": from_m,
            "to_m": to_m,
            "length_m": to_m - from_m,
            "lts": lts if np.isfinite(lts) else np.nan,
            "geometry": substring(geometry, from_m, to_m),
        })

bike_loops_lts_segments = gpd.GeoDataFrame(segments, geometry="geometry", crs=bike_loops_lts.crs)

print(f"{len(bike_loops_lts_segments):,} segments on {len(bike_loops_lts):,} links")

In [ ]:
# Check: the segments of each link add up to the link's length
length_difference = (
    bike_loops_lts_segments.groupby("link_id")["length_m"].sum()
    - bike_loops_lts.geometry.length
)
print(f"Largest difference between summed segments and link length: {length_difference.abs().max():.6f} m")

print("Segments per link:")
display(bike_loops_lts_segments.groupby("link_id").size().describe().round(1).to_frame("segments"))

print("Segment lengths (m):")
display(bike_loops_lts_segments["length_m"].describe().round(1).to_frame("length_m"))

print("Total length per LTS (km):")
display(
    bike_loops_lts_segments.groupby("lts", dropna=False)["length_m"].sum()
    .div(1000).round(1).to_frame("km")
)

# Example: the link with the most segments
example_link_id = bike_loops_lts_segments.groupby("link_id").size().idxmax()
print(f"Example, link_id {example_link_id}:")
display(
    bike_loops_lts_segments.loc[bike_loops_lts_segments["link_id"] == example_link_id]
    .drop(columns="geometry").round(1)
)

In [ ]:
if SHOW_INTERACTIVE_MAP:
    # Fixed LTS 1–4 colours stay consistent even if a class is absent.
    lts_map = bike_loops_lts_segments.explore(
        column="lts",
        categorical=True,
        categories=[1, 2, 3, 4],
        cmap=["#0072B2", "#7B3294", "#E66101", "#B2182B"],
        tiles="OpenStreetMap",
        style_kwds={"weight": 4, "opacity": 1},
        highlight_kwds={"weight": 7},
        missing_kwds={"color": "#555555", "label": "No match"},
        tooltip=["lts", "length_m"],
        tooltip_kwds={"aliases": ["LTS score", "Segment length (m)"], "localize": True},
    )
    display(lts_map)


In [ ]:
# Plot 3: bikenwloops links with every LTS along them (the segments above, split wherever the LTS changes).
# km = segment lengths per LTS class; "not covered" = stretches with no dk_bicycle_network edge within TOLERANCE_M.
plot_lts_map(
    bike_loops_lts_segments,
    column="lts",
    title=f"{SAMPLE.title()}: LTS along each bike-loop link",
    km=km_per_lts_class(bike_loops_lts_segments["lts"], bike_loops_lts_segments["length_m"]),
    legend_title="LTS along links",
    output_path=PLOTS_DIR / f"{SAMPLE}_3_lts_along_links.png",
    background=None,          # 2 million dk_bicycle_network edges would turn the whole country grey
    linewidth=0.6,
    figsize=DENMARK_FIGSIZE,
    widen_right=DENMARK_WIDEN_RIGHT,
    shift=BORNHOLM_SHIFT,
    legend_bounds=DENMARK_LEGEND_BOUNDS,
)

In [ ]:
# Save only the selected setting, as in the original notebook.
# Convert pandas string columns for compatibility with older Fiona versions.
for frame, output_path in [(bike_loops_lts, OUTPUT_PATH), (bike_loops_lts_segments, OUTPUT_SEGMENTS_PATH)]:
    output = frame.copy()
    
    for column in output.columns:
        if isinstance(output[column].dtype, pd.StringDtype):
            output[column] = output[column].astype(object)

    output.to_file(output_path, driver="GPKG")
    print("Saved:", output_path)

#### Length-weighted LTS per link

A continuous LTS score per link: each part's LTS counts in proportion to its length.

$$\text{weighted\_lts} = \frac{\sum_i \text{lts}_i \cdot \text{length}_i}{\sum_i \text{length}_i}$$

- The sums only run over **matched** parts. Unmatched stretches (empty `lts`) are left out of both the top and the bottom, so they don't pull the score towards 0.
- Example: 3 km at LTS 1 and 1 km at LTS 4 gives (1·3000 + 4·1000) / 4000 = 1.75.
- The score is always between 1 and 4. It is an average, so it hides short bad stretches: read it together with `max_lts`.


In [ ]:
matched_parts = bike_loops_lts_segments.dropna(subset=["lts"])

lts_times_length = (matched_parts["lts"] * matched_parts["length_m"]).groupby(matched_parts["link_id"]).sum()
matched_length = matched_parts.groupby("link_id")["length_m"].sum()

# Links with no matched parts get NaN
bike_loops_lts["weighted_lts"] = bike_loops_lts["link_id"].map(lts_times_length / matched_length)

display(bike_loops_lts[["link_id", "edge_id", "mm_len", "max_lts", "weighted_lts", "coverage_pct"]].head(10).round(2))

In [ ]:
# Check: the same score from the point shares (lts1_pct ... lts4_pct) should be almost identical.
# They differ only because the parts use exact lengths, while every point counts as SAMPLING_LENGTH_M.
# That matters on very short links with only a few points (on Lolland-Falster: up to ~0.3 on links under 300 m).
weighted_lts_from_points = (
    1 * bike_loops_lts["lts1_pct"]
    + 2 * bike_loops_lts["lts2_pct"]
    + 3 * bike_loops_lts["lts3_pct"]
    + 4 * bike_loops_lts["lts4_pct"]
) / bike_loops_lts["coverage_pct"]
print(f"Largest difference to the point-based score: {(bike_loops_lts['weighted_lts'] - weighted_lts_from_points).abs().max():.3f}")

print("Weighted LTS:")
display(bike_loops_lts["weighted_lts"].describe().round(2).to_frame("weighted_lts"))

print("Weighted LTS grouped by max LTS (how bad is the average when the worst stretch is 1, 2, 3, 4?):")
display(bike_loops_lts.groupby("max_lts")["weighted_lts"].describe().round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5), facecolor="#fffffb")

ax.hist(bike_loops_lts["weighted_lts"].dropna(), bins=np.arange(1, 4.01, 0.1), color="#16855b")
ax.set_xlabel("Length-weighted LTS")
ax.set_ylabel("Links")
ax.set_xlim(1, 4)
ax.set_title(f"{SAMPLE.title()}: length-weighted LTS per bike-loop link")

fig.savefig(PLOTS_DIR / f"{SAMPLE}_weighted_lts_histogram.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Plot 2: bikenwloops links coloured by their length-weighted LTS (continuous 1–4).
# The legend is a histogram of the score: km of links per WEIGHTED_BIN_WIDTH-wide bin from 1 to 4,
# each bar coloured like the map at the bin's midpoint. A score of exactly 4 goes in the last bin.
WEIGHTED_BIN_WIDTH = 0.25
WEIGHTED_BIN_EDGES = np.linspace(1, 4, round(3 / WEIGHTED_BIN_WIDTH) + 1)

weighted = bike_loops_lts["weighted_lts"].to_numpy()
link_km = bike_loops_lts.geometry.length.to_numpy() / 1000
covered = ~np.isnan(weighted)
bin_index = np.clip(np.digitize(weighted[covered], WEIGHTED_BIN_EDGES) - 1, 0, len(WEIGHTED_BIN_EDGES) - 2)
weighted_km_per_bin = np.bincount(bin_index, weights=link_km[covered], minlength=len(WEIGHTED_BIN_EDGES) - 1)
weighted_not_covered_km = link_km[~covered].sum()

plot_lts_map(
    bike_loops_lts,
    column="weighted_lts",
    title=f"{SAMPLE.title()}: length-weighted LTS per bike-loop link",
    km=None,
    legend_title="Length-weighted LTS",
    output_path=PLOTS_DIR / f"{SAMPLE}_2_weighted_lts.png",
    continuous=True,
    background=None,          # 2 million dk_bicycle_network edges would turn the whole country grey
    linewidth=0.6,
    figsize=DENMARK_FIGSIZE,
    widen_right=DENMARK_WIDEN_RIGHT,
    shift=BORNHOLM_SHIFT,
    legend=lambda ax: add_lts_histogram_legend(
        ax,
        WEIGHTED_BIN_EDGES,
        weighted_km_per_bin,
        weighted_not_covered_km,
        title="Length-weighted LTS",
        xlabel=f"Length-weighted LTS (bins of {WEIGHTED_BIN_WIDTH:g})",
        bounds=DENMARK_LEGEND_BOUNDS,
    ),
)

In [ ]:
# Saved as a separate file, so edges_slope_lts.gpkg from above is not overwritten
OUTPUT_WEIGHTED_PATH = DATA_DIR / "edges_slope_lts_weighted.gpkg"

bike_loops_lts.to_file(OUTPUT_WEIGHTED_PATH, driver="GPKG")
print("Saved:", OUTPUT_WEIGHTED_PATH)

# Tests + analyses

## Sampling length comparison
Run this single cell to calculate and save `sampling_length_comparison.csv` for `SAMPLING_LENGTH_M_LIST`. It does not change `SAMPLING_LENGTH_M`, the selected links/segments, or the maps. It saves no additional maps or per-setting GeoPackages.

In [ ]:
# # This cell alone calculates and saves the sampling-distance comparison.
# # Numeric settings use the original sample-point rule; all-segments mode uses
# # tolerance-buffer intersections (polygonal approximations, quad_segs=64).
# from collections import defaultdict
# from itertools import groupby

# def build_sampling_length_comparison():
#     dk_geometries = dk_edges.geometry.to_numpy()
#     dk_lts = dk_edges["lts"].to_numpy(dtype=float)
#     dk_tree = shapely.STRtree(dk_geometries)
#     road_buffer_cache = {}
#     BUFFER_QUAD_SEGS = 64
#     POINT_BATCH_SIZE = 8192

#     def merge_intervals(intervals):
#         """Merge adjacent equal-score intervals, including adjacent unmatched intervals."""
#         merged = []
#         for start, end, score in intervals:
#             if end <= start:
#                 continue
#             if merged and score == merged[-1][2] and start == merged[-1][1]:
#                 merged[-1] = (merged[-1][0], end, score)
#             else:
#                 merged.append((start, end, score))
#         return merged

#     def sampled_intervals(geometry, spacing):
#         distances = np.arange(0, geometry.length, spacing)
#         scores = np.full(len(distances), np.inf)
#         for offset in range(0, len(distances), POINT_BATCH_SIZE):
#             points = shapely.line_interpolate_point(geometry, distances[offset:offset + POINT_BATCH_SIZE])
#             point_ids, road_ids = dk_tree.query(points, predicate="dwithin", distance=TOLERANCE_M)
#             np.minimum.at(scores, offset + point_ids, dk_lts[road_ids])
#         changes = np.flatnonzero(scores[1:] != scores[:-1]) + 1
#         starts = np.r_[0, changes]
#         ends = np.r_[changes, len(scores)]
#         intervals = [
#             (0.0 if start == 0 else float((distances[start - 1] + distances[start]) / 2),
#              float(geometry.length) if end == len(scores) else float((distances[end - 1] + distances[end]) / 2),
#              float(scores[start]))
#             for start, end in zip(starts, ends)
#         ]
#         return intervals, scores

#     def line_components(geometry):
#         if geometry.geom_type == "LineString":
#             yield geometry
#         elif geometry.geom_type in ("MultiLineString", "GeometryCollection"):
#             for component in geometry.geoms:
#                 yield from line_components(component)

#     def all_segments_intervals(geometry):
#         events = [(0.0, None, 0), (float(geometry.length), None, 0)]
#         candidates = dk_tree.query(geometry, predicate="dwithin", distance=TOLERANCE_M)
#         for road_id in candidates:
#             road_id = int(road_id)
#             if road_id not in road_buffer_cache:
#                 road_buffer_cache[road_id] = shapely.buffer(
#                     dk_geometries[road_id], TOLERANCE_M, quad_segs=BUFFER_QUAD_SEGS
#                 )
#             intersection = geometry.intersection(road_buffer_cache[road_id])
#             for component in line_components(intersection):
#                 start, end = sorted((
#                     float(geometry.project(shapely.Point(component.coords[0]))),
#                     float(geometry.project(shapely.Point(component.coords[-1]))),
#                 ))
#                 start, end = max(0.0, start), min(float(geometry.length), end)
#                 if end > start:
#                     events.extend([(start, float(dk_lts[road_id]), 1), (end, float(dk_lts[road_id]), -1)])
#         events.sort(key=lambda event: event[0])
#         grouped_events = [(distance, list(group)) for distance, group in groupby(events, key=lambda event: event[0])]
#         active = defaultdict(int)
#         intervals = []
#         for i, (start, group) in enumerate(grouped_events[:-1]):
#             for _, score, delta in group:
#                 if score is not None:
#                     active[score] += delta
#             end = grouped_events[i + 1][0]
#             score = min((score for score, count in active.items() if count > 0), default=np.inf)
#             intervals.append((start, end, score))
#         return merge_intervals(intervals)

#     rows = []
#     total_length = bike_loops_edges.length.sum()
#     for setting in SAMPLING_LENGTH_M_LIST:
#         segment_count = 0
#         class_lengths = {score: 0.0 for score in (1, 2, 3, 4)}
#         for geometry in bike_loops_edges.geometry:
#             if setting == "include all segments":
#                 if not geometry.is_simple:
#                     raise ValueError("All-segments comparison requires links without self-intersections.")
#                 intervals = all_segments_intervals(geometry)
#             else:
#                 if not isinstance(setting, (int, float)) or not np.isfinite(setting) or setting <= 0:
#                     raise ValueError(f"Invalid sampling length: {setting!r}")
#                 intervals, _ = sampled_intervals(geometry, float(setting))
#             segment_count += len(intervals)
#             for start, end, score in intervals:
#                 if score in class_lengths:
#                     class_lengths[score] += end - start
#         rows.append({
#             "setting": str(setting),
#             "sampling_length_m": np.nan if setting == "include all segments" else float(setting),
#             "links": len(bike_loops_edges), "segments": segment_count,
#             "total_length_km": total_length / 1000,
#             "matched_length_pct": 100 * sum(class_lengths.values()) / total_length,
#             **{f"lts{score}_length_pct": 100 * length / total_length for score, length in class_lengths.items()},
#         })
#         print(f"{setting}: {segment_count:,} segments", flush=True)
#     return pd.DataFrame(rows)

# comparison = build_sampling_length_comparison()
# COMPARISON_PATH = DATA_DIR / "sampling_length_comparison.csv"
# comparison.to_csv(COMPARISON_PATH, index=False)
# display(comparison.round(3))
# print("Saved:", COMPARISON_PATH)

## Shared regional segment summary
Run this cell to update this region's row in the two shared tables. It reads the saved `sampling_length_comparison.csv` (or `segment_length_comparison.csv`, its name before the rename) and keeps the other regions' rows.

- `region_sampling_length_summary.csv`: **percentages**. Each sampling distance as % of the region's "include all segments" count. The "include all segments" column itself stays a count.
- `region_sampling_length_counts.csv`: the number of segments. The percentages are recalculated from this file for every region, so the table never mixes counts and percentages.
- If the new files don't exist yet, the cell reads the files with the old names (`region_segment_length_counts.csv` / `region_segment_length_summary.csv`), so no regions are lost. The old files are not changed or deleted.


In [ ]:
# # Update this region's segment counts, then recalculate the percentage table for every region.
# # Can be rerun from the saved comparison CSV.
# COMPARISON_PATH = DATA_DIR / "sampling_length_comparison.csv"
# if not COMPARISON_PATH.exists():
#     COMPARISON_PATH = DATA_DIR / "segment_length_comparison.csv"  # name used before the rename
# region_comparison = pd.read_csv(COMPARISON_PATH)
# REGION_SUMMARY_PATH = DATA_OUTPUT_DIR / "region_sampling_length_summary.csv"  # % of baseline
# REGION_COUNTS_PATH = DATA_OUTPUT_DIR / "region_sampling_length_counts.csv"    # number of segments
# # Names used before the rename; only read when the new files don't exist yet
# OLD_REGION_SUMMARY_PATH = DATA_OUTPUT_DIR / "region_segment_length_summary.csv"
# OLD_REGION_COUNTS_PATH = DATA_OUTPUT_DIR / "region_segment_length_counts.csv"
# BASELINE_SETTING = "include all segments"

# # Read the counts table. The first time, take it from the summary file, which held counts before.
# counts_source_path = next((path for path in [REGION_COUNTS_PATH, OLD_REGION_COUNTS_PATH] if path.exists()), None)
# summary_source_path = next((path for path in [REGION_SUMMARY_PATH, OLD_REGION_SUMMARY_PATH] if path.exists()), None)
# if counts_source_path is not None:
#     region_segment_counts = pd.read_csv(counts_source_path, index_col="Region")
# elif summary_source_path is not None:
#     region_segment_counts = pd.read_csv(summary_source_path, index_col="Region")
#     if not (region_segment_counts.dropna(how="all") % 1 == 0).all().all():
#         raise ValueError(f"{summary_source_path.name} does not contain counts; cannot create {REGION_COUNTS_PATH.name} from it.")
# else:
#     region_segment_counts = pd.DataFrame(index=pd.Index([], name="Region"))

# region_name = SAMPLE.replace("_", " ").title()
# region_counts = region_comparison.set_index("setting")["segments"]
# if BASELINE_SETTING not in region_counts.index:
#     raise ValueError("The comparison table is missing the 'include all segments' baseline.")

# # 1) Counts: update (or add) this region's row
# region_columns = list(dict.fromkeys([*region_segment_counts.columns, *region_counts.index]))
# region_rows = list(dict.fromkeys([*region_segment_counts.index, region_name]))
# region_segment_counts = region_segment_counts.reindex(index=region_rows, columns=region_columns)
# region_segment_counts.loc[region_name, region_counts.index] = region_counts.to_numpy()
# region_segment_counts = region_segment_counts.astype("Int64")
# region_segment_counts.index.name = "Region"
# region_segment_counts.to_csv(REGION_COUNTS_PATH)

# # 2) Percentages: every region's counts divided by its own baseline count
# baseline_counts = region_segment_counts[BASELINE_SETTING]
# if baseline_counts.isna().any() or (baseline_counts <= 0).any():
#     raise ValueError("Every region needs a positive 'include all segments' count.")
# region_sampling_length_summary = (
#     region_segment_counts.astype(float)
#     .div(baseline_counts.astype(float), axis=0)
#     .mul(100)
#     .round(2)
# )
# # Keep the reference count; only the sampling-distance columns are percentages.
# region_sampling_length_summary[BASELINE_SETTING] = baseline_counts
# region_sampling_length_summary.to_csv(REGION_SUMMARY_PATH, float_format="%.2f")

# print("Baseline column = number of segments; sampling-distance columns = % of baseline:")
# display(region_sampling_length_summary)
# print("Saved:", REGION_SUMMARY_PATH)
# print("Saved:", REGION_COUNTS_PATH)

In [ ]:
# from matplotlib.ticker import MaxNLocator, MultipleLocator, StrMethodFormatter

# # Keep these charts inline; only the two selected maps are saved to plots/.
# setting_labels = [
#     "All segments" if str(setting) == "include all segments" else f"{float(setting):g}"
#     for setting in comparison["setting"]
# ]
# share_columns = [f"lts{score}_length_pct" for score in (1, 2, 3, 4)]
# shares = comparison.set_index("setting")[share_columns].copy()
# shares.columns = ["LTS 1", "LTS 2", "LTS 3", "LTS 4"]
# unmatched = 100 - shares.sum(axis=1)
# if (unmatched < -1e-6).any():
#     raise ValueError("LTS length shares exceed 100%; check the comparison data.")
# shares["Unmatched"] = unmatched.clip(lower=0)

# with plt.rc_context({"font.size": 10, "axes.titlesize": 13, "axes.labelsize": 10}):
#     fig, axes = plt.subplots(1, 2, figsize=(15, 5.8), layout="constrained")
#     fig.patch.set_facecolor("white")
#     for ax in axes:
#         ax.set_facecolor("white")
#         ax.set_axisbelow(True)  # grid behind, rather than across, the bars
#         ax.grid(axis="y", color="#dce2e7", linewidth=0.8, linestyle="-")
#         ax.grid(axis="x", visible=False)
#         ax.spines[["top", "right"]].set_visible(False)
#         ax.spines[["bottom", "left"]].set_color("#c6cdd3")
#         ax.tick_params(axis="both", length=0, pad=7, labelcolor="#374151")
#         ax.set_xlabel("Sampling distance (m)", labelpad=12)
#         ax.margins(x=0.025)

#     bars = axes[0].bar(
#         range(len(comparison)), comparison["segments"],
#         width=0.68, color="#426d88", edgecolor="white", linewidth=0.6,
#     )
#     axes[0].set_xticks(range(len(setting_labels)), setting_labels)
#     axes[0].set_title("LTS segments by sampling distance", loc="left", weight="bold", pad=18)
#     axes[0].set_ylabel("Number of segments", labelpad=10)
#     axes[0].set_ylim(0, max(1, comparison["segments"].max()) * 1.18)
#     # A few readable integer ticks, instead of a grid line every five segments.
#     axes[0].yaxis.set_major_locator(MaxNLocator(nbins=6, integer=True, steps=[1, 2, 2.5, 5, 10]))
#     axes[0].yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
#     axes[0].bar_label(
#         bars, labels=[f"{int(value):,}" for value in comparison["segments"]],
#         padding=5, fontsize=8, color="#374151",
#     )

#     shares.plot.bar(
#         stacked=True, ax=axes[1], width=0.72, rot=0,
#         color=["#238b45", "#a1d76a", "#fdae61", "#d73027", "#66717e"],
#         edgecolor="white", linewidth=0.35, legend=False,
#     )
#     axes[1].set_xticklabels(setting_labels, rotation=0)
#     axes[1].set_title("Link length by LTS", loc="left", weight="bold", pad=18)
#     axes[1].set_xlabel("Sampling distance (m)", labelpad=12)
#     axes[1].set_ylabel("Share of total link length (%)", labelpad=10)
#     axes[1].set_ylim(0, 100)
#     axes[1].yaxis.set_major_locator(MultipleLocator(20))
#     axes[1].yaxis.set_major_formatter(StrMethodFormatter("{x:.0f}"))
#     axes[1].legend(
#         loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=5,
#         frameon=False, fontsize=9, handlelength=1.2, columnspacing=1.2,
#     )
#     plt.show()

## LTS for edges/segments above threshold

Uses the function for matching LTS to the links from BNN but only considers LTS tags for segments/edges longer than the thresholds and then takes the highest LTS of the remaining segments/edges on that link as the links LTS score.

In [ ]:
# def lts_to_links_threshold(
#     links,
#     osm_edges,
#     length_threshold=LENGTH_THRESHOLD,
#     suffix="",
#     verbose=True
# ):

#     #Drop short OSM edges before building the spatial index
#     edge_lengths = osm_edges.geometry.length
#     osm_kept = osm_edges.loc[edge_lengths >= length_threshold]

#     if verbose:
#         n_dropped = len(osm_edges) - len(osm_kept)
#         print(
#             f"Dropped {n_dropped:,} of {len(osm_edges):,} OSM edges "
#             f"({100 * n_dropped / len(osm_edges):.2f}%) shorter than {length_threshold} m"
#         )

#     spatial_index = osm_kept.sindex
#     geometries = osm_kept.geometry.to_numpy()
#     lts_values = osm_kept["lts"].to_numpy()

#     max_lts_out, coverage_out, shares_out = [], [], []
#     n_links = len(links)

#     max_lts_values, coverage_pcts, lts_shares, _ = match_lts_to_links( #the _ is the point_lts_per_link, which we don't need here
#         links,
#         osm_kept,
#         tolerance_m=TOLERANCE_M,
#         sampling_length_m=SAMPLING_LENGTH_M,
#     )

#     result = links.copy()
#     result[f"max_lts{suffix}"] = max_lts_values
#     result[f"coverage_pct{suffix}"] = coverage_pcts
#     result[[f"lts{i}_pct{suffix}" for i in (1, 2, 3, 4)]] = lts_shares

#     return result, osm_kept   

In [ ]:
# bike_loops_lts_thr, dk_edges_thr = lts_to_links_threshold(
#     bike_loops_edges,
#     dk_edges,
#     length_threshold=LENGTH_THRESHOLD,
# )

# display(bike_loops_lts_thr[["edge_id", "mm_len", "max_lts", "coverage_pct", "lts1_pct", "lts2_pct", "lts3_pct", "lts4_pct"]].describe().round(1))
# display(bike_loops_lts_thr["max_lts"].value_counts(dropna=False).sort_index().to_frame("links"))

In [ ]:
# fig, ax = plt.subplots(figsize=(8, 8), facecolor="#fffffb")

# dk_edges_thr.plot(ax=ax, color="#c8cdd2", linewidth=0.3, alpha=0.6, zorder=1)
# bike_loops_lts_thr.plot(
#     ax=ax,
#     column="max_lts",
#     categorical=True,
#     cmap="RdYlGn_r",
#     linewidth=1.5,
#     legend=True,
#     legend_kwds={"title": "max LTS"},
#     missing_kwds={"color": "black", "label": "no match"},
#     zorder=2,
# )
# ax.set_title(f"{SAMPLE.title()}: worst LTS per bike-loop link (OSM edges ≥ {LENGTH_THRESHOLD} m)")
# ax.set_axis_off()
# fig.savefig(PLOTS_DIR / f"{SAMPLE}_max_lts_min{LENGTH_THRESHOLD}m.png", dpi=300, bbox_inches="tight")
# plt.show()

In [ ]:
# orig = bike_loops_lts["max_lts"]
# thr = bike_loops_lts_thr["max_lts"]

# went_up = thr > orig                      # LTS got worse (higher number)
# went_down = thr < orig                    # LTS got better (lower number)
# lost = orig.notna() & thr.isna()          # had a score before, no match now

# fig, ax = plt.subplots(figsize=(8, 8), facecolor="#fffffb")

# bike_loops_lts_thr.plot(ax=ax, color="#c8cdd2", linewidth=0.8, zorder=1)
# bike_loops_lts_thr.loc[went_up].plot(ax=ax, color="#d62728", linewidth=2, zorder=2)
# bike_loops_lts_thr.loc[went_down].plot(ax=ax, color="#2ca02c", linewidth=2, zorder=2)
# bike_loops_lts_thr.loc[lost].plot(ax=ax, color="black", linewidth=2, zorder=3)

# legend_items = [
#     Line2D([0], [0], color="#c8cdd2", lw=2, label="unchanged"),
#     Line2D([0], [0], color="#d62728", lw=2, label=f"LTS went up ({went_up.sum():,})"),
#     Line2D([0], [0], color="#2ca02c", lw=2, label=f"LTS went down ({went_down.sum():,})"),
#     Line2D([0], [0], color="black", lw=2, label=f"lost LTS score ({lost.sum():,})"),
# ]
# ax.legend(handles=legend_items, title=f"Change with edges ≥ {LENGTH_THRESHOLD} m", loc="best")

# ax.set_title(f"{SAMPLE.title()}: change in max LTS per bike-loop link")
# ax.set_axis_off()
# fig.savefig(PLOTS_DIR / f"{SAMPLE}_max_lts_changed_min{LENGTH_THRESHOLD}m.png", dpi=300, bbox_inches="tight")
# plt.show()


# # Highlight the links that changed without showing the type of change (up, down, lost)
# #fig, ax = plt.subplots(figsize=(8, 8), facecolor="#fffffb")

# #bike_loops_lts_thr.plot(ax=ax, color="#c8cdd2", linewidth=0.8, zorder=1)
# #bike_loops_lts_thr.loc[changed].plot(ax=ax, color="#d62728", linewidth=2, zorder=2)
# #ax.set_title(f"{SAMPLE.title()}: links where max LTS changed ({changed.sum():,})")
# #ax.set_axis_off()
# #fig.savefig(PLOTS_DIR / f"{SAMPLE}_max_lts_changed_min{LENGTH_THRESHOLD}m.png", dpi=300, bbox_inches="tight")
# #plt.show()

In [ ]:
# threshold_rows = []

# for threshold in LENGTH_THRESHOLD_LIST:
#     print(f"--- {threshold} m ---")
#     links_thr, edges_kept = lts_to_links_threshold(
#         bike_loops_edges, dk_edges, length_threshold=threshold, verbose=False
#     )

#     orig = bike_loops_lts["max_lts"]   # no-threshold result, same tolerance and sampling
#     thr = links_thr["max_lts"]

#     n_dropped = len(dk_edges) - len(edges_kept)
#     went_up = (thr > orig).sum()                 # LTS got worse
#     went_down = (thr < orig).sum()               # LTS got better
#     lost = (orig.notna() & thr.isna()).sum()     # had a score before, none now
#     changed_links = ~np.isclose(orig.fillna(-1), thr.fillna(-1))

#     threshold_rows.append({
#         "threshold_m": threshold,
#         "edges_total": len(dk_edges),
#         "edges_dropped": n_dropped,
#         "edges_dropped_pct": 100 * n_dropped / len(dk_edges),
#         "links_no_match": int(thr.isna().sum()),
#         "links_lost_score": int(lost),
#         "links_changed": int(changed_links.sum()),
#         "links_changed_pct": 100 * changed_links.sum() / len(thr),
#         "links_lts_up": int(went_up),
#         "links_lts_down": int(went_down),
#     })

# threshold_comparison = pd.DataFrame(threshold_rows)

# THRESHOLD_COMPARISON_PATH = DATA_DIR / "length_threshold_comparison.csv"
# threshold_comparison.to_csv(THRESHOLD_COMPARISON_PATH, index=False)

# display(threshold_comparison.round(2))
# print("Saved:", THRESHOLD_COMPARISON_PATH)